In [2]:
import glob
import pandas as pd
import re

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kanchana1990/global-food-nutrition-database10k-products")

print("Path to dataset files:", path)

Path to dataset files: /home/mikel/.cache/kagglehub/datasets/kanchana1990/global-food-nutrition-database10k-products/versions/1


In [4]:
glob.glob("/home/mikel/.cache/kagglehub/datasets/kanchana1990/global-food-nutrition-database10k-products/versions/1/*")

['/home/mikel/.cache/kagglehub/datasets/kanchana1990/global-food-nutrition-database10k-products/versions/1/openfoodfacts_nutrition_final_2025-12-10.csv']

In [5]:
food = pd.read_csv('/home/mikel/.cache/kagglehub/datasets/kanchana1990/global-food-nutrition-database10k-products/versions/1/openfoodfacts_nutrition_final_2025-12-10.csv')

In [6]:
food.head()

,code,product_name,brands,countries,quantity,categories,labels,nutriscore_grade,ecoscore_grade,nova_group,ingredients_text,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g
0,6111035000430,Sidi Ali,سيدي علي,Morocco,33 cl,"Beverages and beverages preparations,Beverages...",NaN,a,not-applicable,NaN,OBD1 999 999 1112606 266963207 mb,0.0,0.0,0.0,4.2,1.4,0.0,0.0,0.000000,0.000000
1,6111242100992,perly,perly,"Morocco,United States",100 g,"Dairies,Fermented foods,Fermented milk product...",NaN,unknown,b,3.0,"milk cream, cream, sugar, banana, bacteria",97.0,3.0,NaN,9.4,NaN,NaN,8.0,NaN,NaN
2,6111035002175,Sidi Ali,Sidi Ali,Morocco,2 L,"Beverages and beverages preparations,Beverages...",Green Dot,a,not-applicable,1.0,"Sodium, Calcium, Magnésium, Potassium, Bicarbo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065000,0.026000
3,6111035000058,Eau minérale naturelle,"Les Eaux Minérales d'oulmès,Sidi Ali",Morocco,"1,5 L","Beverages and beverages preparations,Beverages...","ISO 22000,ISO 14001,ISO 45001,ISO 9001",a,not-applicable,1.0,100% mineral water,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065000,0.026000
4,6111252421568,اكوافينا,AQUAFINA,المغرب,33cl,"Boissons et préparations de boissons,Boissons,...",NaN,a,not-applicable,NaN,ouverture et avant le : Voir bouteille. après ...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000508,0.000203


In [7]:
food.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9804 entries, 0 to 9803
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code                9804 non-null   int64  
 1   product_name        9566 non-null   object 
 2   brands              9430 non-null   object 
 3   countries           9802 non-null   object 
 4   quantity            8816 non-null   object 
 5   categories          9683 non-null   object 
 6   labels              7552 non-null   object 
 7   nutriscore_grade    9801 non-null   object 
 8   ecoscore_grade      9802 non-null   object 
 9   nova_group          8959 non-null   float64
 10  ingredients_text    9406 non-null   object 
 11  energy-kcal_100g    9280 non-null   float64
 12  fat_100g            9309 non-null   float64
 13  saturated-fat_100g  9140 non-null   float64
 14  carbohydrates_100g  9284 non-null   float64
 15  sugars_100g         9168 non-null   float64
 16  fiber_

In [8]:
food["countries"].unique().tolist()



['Morocco',
 'Morocco,United States',
 'المغرب',
 "Belgium,Côte d'Ivoire,France,Germany,Guadeloupe,Italy,Luxembourg,Mali,Martinique,New Caledonia,Switzerland,United Kingdom",
 'Democratic Republic of the Congo,Mauritania,Morocco,United States',
 'Maroc,en:morocco',
 'Maroc',
 'France,Germany,Ireland,Italy,Netherlands,Spain,United Kingdom,United States',
 'Algeria,Belgium,France,French Polynesia,Germany,Guadeloupe,Hungary,Luxembourg,Martinique,Morocco,New Caledonia,Réunion,Spain,Switzerland,United States',
 'Algérie,Autriche,Belgique,Bulgarie,Canada,République tchèque,Finlande,France,Polynésie française,Allemagne,Irlande,Italie,Maurice,Maroc,Pays-Bas,Norvège,La Réunion,Roumanie,Singapour,Espagne,Suède,Suisse,Tunisie,Royaume-Uni',
 'Algérie,Belgique,Bulgarie,France,Allemagne,Guadeloupe,Inde,Italie,Luxembourg,Martinique,Maroc,La Réunion,Espagne,Suisse,Tunisie,Royaume-Uni,États-Unis',
 'Marocco',
 'Maroc,États-Unis',
 'Belgium, Bulgaria, France, en:switzerland',
 'Ireland,United Kingdom,Un

In [32]:
import re

def filtrar_potencias_europeas_v3(df, columna='countries'):
    # 1. Definimos los patrones de búsqueda (Regex)
    # He separado Ireland de United Kingdom en el patrón inicial
    patrones = {
        'Spain': r'Spain|España|Espagne|Espanha|Spanien|Spagna|en:es|ES',
        'France': r'France|Francia|Frankreich|Frankrijk|França|en:fr|FR',
        'Germany': r'Germany|Alemania|Allemagne|Deutschland|Duitsland|en:de|DE',
        'United Kingdom': r'United Kingdom|UK|Reino Unido|Royaume-Uni|en:gb|GB|Great Britain',
        'Ireland': r'Ireland|Irlanda|Irlande|Irland|en:ie|IE',
        'Italy': r'Italy|Italia|Italie|Italien|en:it|IT',
        'USA': r'United States|USA|EEUU|en:us|en:united-states',
        'Norway': r'Norway|Noruega|Norvège|Norwegen|Norge|en:no|NO',
        'Portugal': r'Portugal|Portugália|en:pt|PT'
    }

    # Creamos el patrón global uniendo todos con OR (|)
    patron_global = '|'.join(patrones.values())

    # Filtramos el DataFrame original
    df_euro = df[df[columna].str.contains(patron_global, case=False, na=False)].copy()

    # 2. Función de etiquetado con lógica de prioridad
    def etiquetar_pais(row):
        val = str(row).lower()
        
        # Prioridad 1: Irlanda (La ponemos antes para que productos "Ireland, UK" 
        # se puedan clasificar específicamente si así lo deseas, o viceversa)
        if any(term in val for term in ['ireland', 'irlanda', 'irlande', 'en:ie', 'ie:']):
            return 'Ireland'
        
        # Prioridad 2: Reino Unido
        if any(term in val for term in ['united kingdom', 'uk', 'reino unido', 'en:gb', 'gb:', 'gb', 'great britain']):
            return 'United Kingdom'
        
        # Prioridad 3: Francia
        if any(term in val for term in ['france', 'francia', 'frança', 'en:fr', 'fr:']):
            return 'France'
        
        # Prioridad 4: España
        if any(term in val for term in ['spain', 'españa', 'espagne', 'espanha', 'en:es', 'es:']):
            return 'Spain'
        
        # Prioridad 5: Alemania
        if any(term in val for term in ['germany', 'alemania', 'allemagne', 'deutschland', 'en:de', 'de:']):
            return 'Germany'
        
        # Prioridad 6: USA
        if any(term in val for term in ['united states', 'usa', 'eeuu', 'en:us', 'en:united-states']):
            return 'USA'
        
        # Prioridad 7: Italia
        if any(term in val for term in ['italy', 'italia', 'italie', 'en:it', 'it:']):
            return 'Italy'
        
        # Prioridad 8: Noruega
        if any(term in val for term in ['norway', 'noruega', 'norvège', 'en:no', 'no:']):
            return 'Norway'
        
        # Prioridad 9: Portugal
        if any(term in val for term in ['portugal', 'portugália', 'en:pt', 'pt:']):
            return 'Portugal'
        
        return 'Other International'

    df_euro['main_country'] = df_euro[columna].apply(etiquetar_pais)
    
    return df_euro

# Ejecutamos
food_euro = filtrar_potencias_europeas_v3(food)

# Verificación
print("Resultados por país (incluyendo Irlanda):")
print(food_euro['main_country'].value_counts())

Resultados por país (incluyendo Irlanda):
main_country
France                 4566
United Kingdom         1805
Spain                   461
Ireland                 390
Germany                 360
Other International     286
USA                      72
Italy                    41
Portugal                 15
Name: count, dtype: int64


In [33]:
import re

def filtrar_potencias_europeas_balance(df, columna='countries'):
    # 1. Definimos los patrones de búsqueda (Regex)
    patrones = {
        'Spain': r'Spain|España|Espagne|Espanha|Spanien|Spagna|en:es|ES',
        'France': r'France|Francia|Frankreich|Frankrijk|França|en:fr|FR',
        'Germany': r'Germany|Alemania|Allemagne|Deutschland|Duitsland|en:de|DE',
        'United Kingdom': r'United Kingdom|UK|Reino Unido|Royaume-Uni|en:gb|GB|Great Britain',
        'Ireland': r'Ireland|Irlanda|Irlande|Irland|en:ie|IE',
        'Italy': r'Italy|Italia|Italie|Italien|en:it|IT',
        'USA': r'United States|USA|EEUU|en:us|en:united-states',
        'Norway': r'Norway|Noruega|Norvège|Norwegen|Norge|en:no|NO',
        'Portugal': r'Portugal|Portugália|en:pt|PT'
    }

    # Creamos el patrón global
    patron_global = '|'.join(patrones.values())

    # 2. Filtrado inicial (Dataframe de trabajo)
    df_result = df[df[columna].str.contains(patron_global, case=False, na=False)].copy()

    # 3. Función de etiquetado
    def etiquetar_pais(row):
        val = str(row).lower()
        if any(term in val for term in ['ireland', 'irlanda', 'irlande', 'en:ie', 'ie:']):
            return 'Ireland'
        if any(term in val for term in ['united kingdom', 'uk', 'reino unido', 'en:gb', 'gb:', 'gb', 'great britain']):
            return 'United Kingdom'
        if any(term in val for term in ['france', 'francia', 'frança', 'en:fr', 'fr:']):
            return 'France'
        if any(term in val for term in ['spain', 'españa', 'espagne', 'espanha', 'en:es', 'es:']):
            return 'Spain'
        if any(term in val for term in ['germany', 'alemania', 'allemagne', 'deutschland', 'en:de', 'de:']):
            return 'Germany'
        if any(term in val for term in ['united states', 'usa', 'eeuu', 'en:us', 'en:united-states']):
            return 'USA'
        if any(term in val for term in ['italy', 'italia', 'italie', 'en:it', 'it:']):
            return 'Italy'
        if any(term in val for term in ['norway', 'noruega', 'norvège', 'en:no', 'no:']):
            return 'Norway'
        if any(term in val for term in ['portugal', 'portugália', 'en:pt', 'pt:']):
            return 'Portugal'
        return 'Other International'

    # Creamos la columna
    df_result['main_country'] = df_result[columna].apply(etiquetar_pais)

    # --- CÁLCULO DE TOTALES ---
    total_original = len(df) # Total de filas antes del filtro
    total_en_potencias = len(df_result[df_result['main_country'] != 'Other International'])
    total_en_others = len(df_result[df_result['main_country'] == 'Other International'])
    
    # Filas que el filtro inicial NO capturó (pérdida total de datos geográficos)
    total_fuera_de_estudio = total_original - len(df_result)
    
    porcentaje_cobertura = (total_en_potencias / total_original) * 100

    print("-" * 40)
    print(f"BALANCE DE CLASIFICACIÓN GEOGRÁFICA:")
    print(f"Total de productos en dataset: {total_original}")
    print(f"Ubicados en potencias clave: {total_en_potencias}")
    print(f"Ubicados en 'Other International': {total_en_others}")
    print(f"Sin información geográfica útil: {total_fuera_de_estudio}")
    print(f"Porcentaje de éxito (Potencias): {porcentaje_cobertura:.2f}%")
    print("-" * 40)

    return df_result

# Ejecución
food_euro = filtrar_potencias_europeas_balance(food)

----------------------------------------
BALANCE DE CLASIFICACIÓN GEOGRÁFICA:
Total de productos en dataset: 7996
Ubicados en potencias clave: 7710
Ubicados en 'Other International': 286
Sin información geográfica útil: 0
Porcentaje de éxito (Potencias): 96.42%
----------------------------------------


In [10]:
# Ver los 20 valores de 'countries' más frecuentes que cayeron en 'Other'
print(food[food['main_country'] == 'Other']['countries'].value_counts().head(20))

Series([], Name: count, dtype: int64)


In [11]:
food.sample(20)

,code,product_name,brands,countries,quantity,categories,labels,nutriscore_grade,ecoscore_grade,nova_group,...,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,main_country
5533,8000320990113,Passata,Cirio,"Bulgaria, Germany, Switzerland, United Kingdom...",500g,"Plant-based foods and beverages, Plant-based f...","FSC,FSC Mix,Made in Italy,Climate Neutral product",a,a,3.0,...,35.0,0.10,0.000,6.3,4.4,1.1,1.20,0.350,0.140,United Kingdom
786,20337292,Pistacchi,AlestoLidl,"France,Allemagne,Italie,Espagne",250g,"Aliments et boissons à base de végétaux,Alimen...","Peu ou pas de sel,Végétarien,Végétalien,Point ...",a,d,3.0,...,605.0,49.20,6.300,10.3,5.6,7.6,26.50,0.030,0.012,France
2769,3270190021087,Fromage Blanc Nature 3%,Carrefour,France,1 kg,"Produits laitiers, Produits fermentés, Dessert...","Source de protéines,Source de calcium,Lait Fra...",a,a,3.0,...,71.0,3.00,1.900,4.5,4.4,NaN,6.50,0.130,0.052,France
8121,96619676811,Almond Butter,Kirkland Signature,"Bolivia,Francia,México,España,Estados Unidos",765 g (27 oz) 1.68 lb,"Alimentos y bebidas de origen vegetal,Alimento...","Sin gluten,Kosher,en:Certified gluten-free,Uni...",a,unknown,1.0,...,654.0,55.00,4.500,22.0,3.4,7.3,19.00,0.030,0.012,France
3574,3103220035214,Croco,Haribo,"Francia,Suiza",280 g,"Snacks, Sweet snacks, Confectioneries, Candies...","Green Dot, Natural colorings",d,b,4.0,...,345.0,0.50,0.100,79.0,51.0,NaN,5.70,0.030,0.012,France
7086,332071,Baked Beans (Reduced Sugar & Salt),Sainsburys,United Kingdom,NaN,"Plant-based foods and beverages, Plant-based f...",NaN,a,a-plus,4.0,...,154.0,0.90,0.200,21.0,4.9,11.9,9.60,0.600,0.240,United Kingdom
5939,4008258065013,Mandeln,Seeberger,"France,Allemagne,Luxembourg,Suisse",200g,"Pflanzliche Lebensmittel und Getränke, Pflanzl...",Triman,a,d,1.0,...,619.0,53.00,4.100,5.7,3.4,11.2,24.00,0.010,0.004,France
115,8076809513753,Pesto alla Genovese,Barilla,"Australie,Belgique,France,Allemagne,Maroc,Norv...",190 g e,"Condiments,Sauces,Sauces pour pâtes,Sauces Pes...","Sans gluten,AOP,Triman,ISCC Plus,Produit en It...",e,c,4.0,...,492.0,47.00,5.300,11.0,5.0,3.0,4.70,3.200,1.280,France
3888,4056489546085,Bio crackers,maître Jean pierre,France,200 g,"snacks, snacks-sales, amuse-gueules, biscuits-...","Organic, EU Organic, Source of fibre, High fib...",d,a-plus,3.0,...,442.0,18.00,4.800,47.0,2.4,10.0,18.00,1.800,0.720,France
7444,3281780063509,Ma Salade de Lentilles à la Lyonnaise,Pierre Martinet,"France,en:france",300 g,"Aliments et boissons à base de végétaux, Alime...","Point Vert, Nutriscore, Nutriscore A",a,unknown,4.0,...,122.0,4.20,0.300,12.0,1.2,5.4,6.40,0.780,0.312,France


In [12]:
food.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8733 entries, 1 to 9803
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code                8733 non-null   int64  
 1   product_name        8571 non-null   object 
 2   brands              8623 non-null   object 
 3   countries           8733 non-null   object 
 4   quantity            8135 non-null   object 
 5   categories          8696 non-null   object 
 6   labels              7199 non-null   object 
 7   nutriscore_grade    8732 non-null   object 
 8   ecoscore_grade      8732 non-null   object 
 9   nova_group          8299 non-null   float64
 10  ingredients_text    8588 non-null   object 
 11  energy-kcal_100g    8458 non-null   float64
 12  fat_100g            8474 non-null   float64
 13  saturated-fat_100g  8358 non-null   float64
 14  carbohydrates_100g  8463 non-null   float64
 15  sugars_100g         8380 non-null   float64
 16  fiber_100g 

In [13]:
food["categories"].unique().tolist()

['Dairies,Fermented foods,Fermented milk products,Snacks,Desserts,Dairy desserts,Fermented dairy desserts,Plain fermented dairy desserts,Plain fermented dairy desserts with cream',
 'Beverages and beverages preparations,Beverages,Waters,Spring waters,Mineral waters,Unsweetened beverages,Natural mineral waters',
 'Dairies,Milks,Homogenized milks,UHT Milks,Whole milks,Cow milks,Whole milk UHT',
 'Yeast extract spreads',
 'Plant-based foods and beverages,Dairies,Plant-based foods,Fats,Spreads,Plant-based spreads,Salted spreads,Vegetable fats,Spreadable fats,Animal fats,Dairy spreads,Margarines,Milkfat,Butters',
 'Snacks,Breakfasts,Sweet snacks,Biscuits and cakes,Biscuits and crackers,Sandwich biscuits',
 'Snacks,Snacks sucrés,Cacao et dérivés,Chocolats,Chocolats noirs,Chocolats noirs en tablette,Chocolats noirs extra fin',
 'Petit-déjeuners,Produits à tartiner,Produits à tartiner sucrés,Pâtes à tartiner,Pâtes à tartiner aux noisettes,Pâtes à tartiner au chocolat,Pâtes à tartiner aux noise

In [14]:
def unificacion_quirurgica(df, columna='categories'):
    def limpiar(txt):
        if pd.isna(txt): return "Otros"
        txt = re.sub(r'[a-z]{2}:', '', str(txt)).lower()
        
        # EL ORDEN AQUÍ ES CRÍTICO: 
        # Buscamos primero lo que suele ser "ruido" en las etiquetas de bebidas
        
        # 1. Proteína Animal (Muy específico)
        if any(p in txt for p in ['meat', 'viande', 'carne', 'fish', 'poisson', 'pescado', 'atún', 'thon', 'sardine']):
            return 'meat_fish'
            
        # 2. Lácteos (Antes de bebidas para no confundir leche con refresco)
        if any(p in txt for p in ['dairy', 'laitier', 'lácteo', 'cheese', 'fromage', 'queso', 'yogurt', 'yaourt', 'milk', 'lait', 'leche']):
            return 'dairy_eggs'
            
        # 3. Dulces y Snacks (Muchos snacks vegetales se marcaban como bebidas)
        if any(p in txt for p in ['snack', 'biscuits', 'gateaux', 'galletas', 'chocolat', 'candy', 'confectionery', 'sweet', 'sucré']):
            return 'snacks_sweets'
            
        # 4. Grasas y Salsas
        if any(p in txt for p in ['oil', 'huile', 'aceite', 'sauce', 'condiment', 'ketchup', 'pesto', 'spread', 'tartiner', 'untable']):
            return 'fats_sauces'
            
        # 5. Cereales y Pan (Para evitar que el desayuno se vaya a bebidas)
        if any(p in txt for p in ['céréale', 'cereals', 'bread', 'pain', 'pan', 'pão', 'pasta', 'pâte', 'muesli', 'breakfast', 'petit-déjeuner']):
            return 'cereals_bread'

        # 6. BEBIDAS (Ahora sí, solo si no es nada de lo anterior)
        # Excluimos la frase "plant-based foods and beverages" de la búsqueda de la palabra 'beverage'
        txt_sin_ruido = txt.replace('plant-based foods and beverages', '')
        if any(p in txt_sin_ruido for p in ['beverage', 'boisson', 'bebida', 'bevande', 'water', 'eau', 'agua', 'juice', 'jus', 'zumo', 'soda', 'cola', 'coffee', 'café']):
            return 'beverage'
            
        # 7. Vegetales puros (Frutas y verduras frescas)
        if any(p in txt for p in ['vegetal', 'plant-based', 'legume', 'fruit', 'fruta', 'légume', 'verdura', 'tomato', 'tomate']):
            return 'plant_based'
            
        return "Otros"

    df['category_unified'] = df[columna].apply(limpiar)
    return df

food = unificacion_quirurgica(food)
print(food['category_unified'].value_counts())

category_unified
snacks_sweets    2344
dairy_eggs       1547
cereals_bread    1397
fats_sauces      1049
beverage          827
Otros             718
meat_fish         562
plant_based       289
Name: count, dtype: int64


In [30]:
food.sample()

,code,product_name,brands,countries,quantity,categories,labels,nutriscore_grade,ecoscore_grade,nova_group,...,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,main_country,category_unified,health_vs_proc,corporacion
6301,8424790111005,Kéfir,Pastoret,Espanha,500g,"Bebidas,Laticínios,Alimentos Fermentados,Produtos lácteos fermentados,Bebidas lácteas,Bebidas fermentadas,Bebidas lácteas fermentadas,Kefir","Sem glúten,Ponto Verde,es:Recicla azul",c,Unknown,1.0,...,3.8,3.8,NaN,3.4,0.1,0.04,Spain,dairy_eggs,c1.0,Otras / Local


In [16]:
# 1. Unificar Nutriscore y Ecoscore a minúsculas
food['nutriscore_grade'] = food['nutriscore_grade'].str.replace('not-applicable', 'unknown').str.lower()
food['ecoscore_grade'] = food['ecoscore_grade'].str.replace('unknown', 'u').str.lower()

# 2. Eliminar filas donde NOVA es NaN (lo necesitamos para las hipótesis)
food = food.dropna(subset=['nova_group'])

# 3. Eliminar filas con 'Other' en main_country (para que la comparativa sea solo de los 4 países)
food = food[food['main_country'] != 'Other']

# 4. Asegurar que las grasas saturadas no sean superiores a las grasas totales (error de datos)
food = food[food['saturated-fat_100g'] <= food['fat_100g']]

In [17]:
print(food['category_unified'].value_counts())

category_unified
snacks_sweets    2212
dairy_eggs       1421
cereals_bread    1324
fats_sauces       980
beverage          664
Otros             577
meat_fish         547
plant_based       271
Name: count, dtype: int64


In [18]:
# 1. Conteo detallado de Ecoscore
print("--- DISTRIBUCIÓN DE ECOSCORE ---")
print(food['ecoscore_grade'].value_counts())

# 2. Conteo de países (para ver el equilibrio de tu muestra)
print("\n--- PRODUCTOS POR PAÍS ---")
print(food['main_country'].value_counts())

# 3. (Opcional) Si decides unificar el a-plus con el a para simplificar:
# food_med['ecoscore_grade'] = food_med['ecoscore_grade'].replace('a-plus', 'a')

--- DISTRIBUCIÓN DE ECOSCORE ---
ecoscore_grade
b                 1546
a                 1381
u                 1139
c                 1133
d                 1126
e                  795
a-plus             479
not-applicable     220
f                  177
Name: count, dtype: int64

--- PRODUCTOS POR PAÍS ---
main_country
France                 4769
United Kingdom         2160
Spain                   469
Germany                 240
Other International     232
USA                      71
Italy                    41
Portugal                 13
Norway                    1
Name: count, dtype: int64


In [19]:
# Unificamos la escala para que tus gráficos de barras salgan ordenados
mapping_eco = {
    'a-plus': 'A+',
    'a': 'A',
    'b': 'B',
    'c': 'C',
    'd': 'D',
    'e': 'E',
    'f': 'E', # Corregimos el error de la F mandándola a la E
    'u': 'Unknown',
    'not-applicable': 'Unknown'
}

food['ecoscore_grade'] = food['ecoscore_grade'].map(mapping_eco)

In [20]:
# 1. Unificamos a minúsculas por si hay mezclas de 'A' y 'a'
food['nutriscore_grade'] = food['nutriscore_grade'].str.lower()

# 2. Conteo de Nutri-Score por País
print("--- DISTRIBUCIÓN DE NUTRISCORE POR PAÍS ---")
tab_nutri = pd.crosstab(food['main_country'], food['nutriscore_grade'], margins=True)
print(tab_nutri)

# 3. Porcentaje de 'A' y 'B' (Calidad Superior) por país
print("\n--- % DE PRODUCTOS SALUDABLES (A o B) ---")
nutri_ab = food[food['nutriscore_grade'].isin(['a', 'b'])]
print(nutri_ab['main_country'].value_counts(normalize=True) * 100)

--- DISTRIBUCIÓN DE NUTRISCORE POR PAÍS ---
nutriscore_grade        a     b     c     d     e  unknown   All
main_country                                                    
France                688   658  1248   971   979      225  4769
Germany                32    25    51    63    53       16   240
Italy                   9     2    10    10    10        0    41
Norway                  0     0     0     1     0        0     1
Other International    43    28    53    55    37       16   232
Portugal                4     0     3     0     6        0    13
Spain                  77    61   126   113    64       28   469
USA                    10    13    19    13     8        8    71
United Kingdom        463   297   517   393   352      138  2160
All                  1326  1084  2027  1619  1509      431  7996

--- % DE PRODUCTOS SALUDABLES (A o B) ---
main_country
France                 55.850622
United Kingdom         31.535270
Spain                   5.726141
Other International  

In [21]:
food.shape

(7996, 22)

In [22]:
# 1. Tabla cruzada de Países vs NOVA
print("--- GRADO DE PROCESAMIENTO (NOVA) POR PAÍS ---")
tab_nova = pd.crosstab(food['main_country'], food['nova_group'], margins=True)
print(tab_nova)

# 2. % de Ultraprocesados (NOVA 4) sobre el total de cada país
nova_4_pct = food[food['nova_group'] == 4]['main_country'].value_counts() / food['main_country'].value_counts() * 100
print("\n--- % DE ULTRAPROCESADOS (NOVA 4) POR PAÍS ---")
print(nova_4_pct.sort_values(ascending=False))

--- GRADO DE PROCESAMIENTO (NOVA) POR PAÍS ---
nova_group           1.0  2.0   3.0   4.0   All
main_country                                   
France               482  202  1129  2956  4769
Germany               24   17    65   134   240
Italy                  7    0     8    26    41
Norway                 0    0     0     1     1
Other International   24   10    74   124   232
Portugal               2    3     2     6    13
Spain                 47   10   110   302   469
USA                   11    3    20    37    71
United Kingdom       280   96   694  1090  2160
All                  877  341  2102  4676  7996

--- % DE ULTRAPROCESADOS (NOVA 4) POR PAÍS ---
main_country
Norway                 100.000000
Spain                   64.392324
Italy                   63.414634
France                  61.983644
Germany                 55.833333
Other International     53.448276
USA                     52.112676
United Kingdom          50.462963
Portugal                46.153846
Name: coun

In [23]:
# Creamos una columna combinada para el análisis
food['health_vs_proc'] = food['nutriscore_grade'] + food['nova_group'].astype(str)

# Filtramos solo los "A4" (Los falsos saludables)
falsos_saludables = food[(food['nutriscore_grade'] == 'a') & (food['nova_group'] == 4)]

print(f"\nDetectados {len(falsos_saludables)} productos 'Falsos Saludables' (Nutri-Score A + NOVA 4)")
print(falsos_saludables['main_country'].value_counts())


Detectados 484 productos 'Falsos Saludables' (Nutri-Score A + NOVA 4)
main_country
France                 252
United Kingdom         158
Spain                   42
Other International     15
Germany                 10
USA                      5
Portugal                 1
Italy                    1
Name: count, dtype: int64


In [24]:
print("--- CONTEO POR GRUPO NOVA ---")
print(food['nova_group'].value_counts().sort_index())

--- CONTEO POR GRUPO NOVA ---
nova_group
1.0     877
2.0     341
3.0    2102
4.0    4676
Name: count, dtype: int64


In [25]:
falsos_saludables.sample()

,code,product_name,brands,countries,quantity,categories,labels,nutriscore_grade,ecoscore_grade,nova_group,...,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,main_country,category_unified,health_vs_proc
7428,3175681225084,"Les Tendres Carrés blé, petits pois, courgette...",Céréal Bio,France,200g,"Meat alternatives, Vegetarian patties, Seitan ...","Organic,Vegetarian,EU Organic,No preservatives...",a,A+,4.0,...,0.9,19.0,3.5,4.1,13.0,0.0025,0.001,France,meat_fish,a4.0


In [26]:
# 1. Filtramos el subset de "Falsos Saludables" (Nutri-Score A + NOVA 4)
falsos_saludables = food[(food['nutriscore_grade'] == 'a') & (food['nova_group'] == 4)].copy()

# 2. Mostramos solo el nombre del producto y la lista de ingredientes
# He añadido 'brands' (marca) porque ayuda mucho a identificar el producto
pd.set_option('display.max_colwidth', None) # Para que no corte el texto de los ingredientes
investigacion_ingredientes = falsos_saludables[['product_name', 'brands', 'ingredients_text']]

# 3. Visualizamos los primeros 20 para tu análisis
investigacion_ingredientes.sample(20)

,product_name,brands,ingredients_text
6026,"Céréales Muesli chocolat & amandes, 400g",U,"Flocons d'AVOINE complets 85,5%, huile de tournesol, copeaux de chocolat 6% (sucre, pate de cacao, beurre de cacao, lecithines [tournesol], arome naturel), AMANDES 4%, miel, melasse, antioxydant: extrait riche en tocopherols. traces eventuelles d'autres cereales contenant du gluten, d'autres fruits a coque, de soja et de lait. avoine ue / non ue. cacao et amande non ue."
6471,Vegetarian 1/4lb Burgers,Linda McCartney,"Rehydrated textured SOYA protein (71%), rapeseed oil, onion, chickpea flour, flavouring, yeast extract, stabiliser: methyl cellulose; garlic purée, malted BARLEY extract, onion powder, salt, garlic powder."
3000,Hipro,Danone,"Lait écrémé, banane (5%), édulcorants (acésulfame-k, sucralose), épaississants (amidon transformé de maïs, pectine de fruits), colorant (caramel ordinaire), jus de citron concentré, arômes, ferments lactiques (lait). Lait : Origine France."
6881,KP Dry Roasted Peanuts,KP,"_Peanuts_, Dry Roasted Flavour (Salt, Flavour Enhancer (Monosodium Glutamate), Dried Yeast Extract, Rice Flour, Paprika, Spice & Herbs, _Celery_ Seed Powder, Dried Onion, Dried Garlic, Colours (Paprika Extract, Turmeric Extract), Smoke Flavouring), Stabiliser (Acacia Gum)."
8271,Cannellini Beans in Water,Four Seasons,"Cannellini Beans, Water, Firming Agent: Calcium Chloride, Antioxidant So ALLERGY ADVICE: For allergens, see ingredients in bold. CAUTION: Care should be taken when opening and disposing of all cans NUTRITION TYPICAL VALUES (drained) Energy Fat"
5160,Cream of Tomato Soup,Aldi Bramwells,"tomatoes (89%), water, modified maize starch, sugar, rapeseed oil, cream (𝐌𝐢𝐥𝐤), salt, dried skimmed 𝐌𝐢𝐥𝐤, flavourings, onion extract, acidity regulatory: citric acid,"
6192,Sunflower and Pumpkin Soft Farmhouse,Sainsburys Taste the Difference,"Fortified British Wheat Flour (Wheat Flour, Calcium Carbonate, Iron, Niacin, Thiamin), Water, Wholemeal British Wheat Flour, Sunflower Seeds (11%), Pumpkin Seeds (6%), Wheat Gluten, Wheat Fibre, Yeast, Wheat Bran, Fermented Wheat Flour, Salt, Oat Flakes, Rapeseed Oil, Soya Flour, Spirit Vinegar, Palm Oil, Flour Treatment Agent: Ascorbic Acid."
8178,Yaourt nature,Carrefour,"_Lait_ partiellement écrémé à 11g/l de matière grasse (origine France), _lactose_ et protéines de _lait_, ferments _lactiques_."
3191,"Le Gourmand Soja, Poivre et Persil","Nestlé, Garden gourmet","Eau, concentré de protéines de SOJA 14,8%, chapelure (farine de BLÉ, eau, huile de colza, levure, sel, extrait de paprika), protéines de BLÉ 5%, oignons déshydratés, huiles végétales (colza, tournesol) en proportion variable, blanc d'ŒUF en poudre 3,9%, purée de pomme, correcteur d'acidité : vinaigre tamponné ; arômes (BLÉ), amidon de maïs, persil 1,1%, flocons de pomme de terre, extrait de levure, coriandre, ail en poudre, fibre et amidon de pois, extrait de malt d'ORGE, sel, vinaigre d'alcool, poivre noir 0,13%, cumin, macis. Peut contenir : SÉSAME, CÉLERI et MOUTARDE."
6388,Danone ALPRO® Joghurt Soja Natur 400g,Alpro,"SOJABASIS (Wasser, geschälte SOJABOHNEN (10,7%)), Zucker, Tricalciumcitrat, Stabilisator (Pektine), Säureregulatoren (Natriumcitrate, Citronensäure), Meersalz, Aroma, Antioxidationsmittel (stark tocopherolhaltige Extrakte, Fettsäureester der Ascorbinsäure), Vitamine (B12, D2), Joghurtkulturen (Str. thermophilus, L. bulgaricus)."


In [27]:
lista_marcas = food["brands"].unique().tolist()

In [28]:
import pandas as pd
import numpy as np

def clasificar_multinacionales(df_columna):
    """
    Recibe una columna de marcas y devuelve un DataFrame con la clasificación.
    """
    # 1. Obtener valores únicos para procesar menos datos y ganar velocidad
    lista_marcas_unicas = df_columna.unique()

    # Diccionario de referencia actualizado (incluye Hacendado y correcciones)
    mapping = {
    # ======================
    # GRANDES MULTINACIONALES
    # ======================
    'Nestlé': [
        'nestle', 'nescafe', 'nescafé', 'nesquik', 'maggi', 'chocapic',
        'garden gourmet', 'le chocolat', 'lion', 'kitkat', 'after eight',
        'cookie crisp', 'golden grahams', 'herta', 'la laitiere', 'la laitière',
        'la laitiere, la laitière', 'mousline', 'vittel', 'perrier', 'badoit',
        'solis', 'buino', 'ricore', 'dolce gusto', 'cailler', 'ovomaltine',
        'ovo maltine', 'cappuccino, gold cappuccino, nescafé'
    ],

    'Unilever': [
        'unilever', 'marmite', 'hellmann', "hellman's", 'hellmanns',
        'knorr', 'knoor', 'amora', 'maizena', 'ben & jerry', 'ben & jerry\'s',
        'maille', 'frigo', 'ligeresa', 'calve', 'calvé', 'flora', 'planta fin',
        'flora proactiv', "fruit d'or", "fruit d'or proactiv, pro activ, proactiv, proactiv expert",
        "fruit d'or,becel", "fruit d'or proactiv,proactiv,proactiv expert,pro activ",
        'lipton', 'benedicta', 'bénédicta', 'skip', 'dove',
        'pukka', 'fruittare', 'the vegetarian butcher', 'magnum', 'magnum,miko'
    ],

    'Mondelez': [
        'mondelez', 'lu', 'lu ', ' lu', 'lu,', 'lu.', 'milka', 'oreo',
        'toblerone', 'tuc', 'belvita', 'cadbury', 'philadelphia',
        'cote d’or', "cote d'or", "côte d'or", 'granola', 'heudebert',
        'pelletier', 'mikado', 'fontaneda', 'principe', 'príncipe', 'barny',
        'oscar mayer', 'belin', 'royal', 'daunat', 'jacob', 'mcvitie', 'nippon'
    ],

    'Danone': [
        'danone', 'activia', 'evian', 'volvic', 'alpro', 'hipro',
        'savane', 'font vella', 'lanjarón', 'les 2 vaches', 'oatly',
        'la salvetat', 'salvetat', 'vrai', 'charles & alice', 'michel & augustin',
        'michel et augustin', 'siggi', 'nyakas', 'danao'
        # yoplait: lo dejamos fuera por solapamiento histórico
    ],

    'Ferrero': [
        'ferrero', 'nutella', 'kinder', 'rocher', 'tic tac', 'tictac',
        'raffaello', 'mon cheri', 'mon chéri', 'mon chéri', "MON CHÉRI"
    ],

    'PepsiCo': [
        'pepsico', 'pepsi', 'lays', "lay's", 'lay&#039;s', 'doritos', 'cheetos',
        'alvalle', 'quaker', 'gatorade', 'tropicana', 'cruesli', 'matutano',
        'benenuts', 'sun bites', 'sun bites'
    ],

    'Coca-Cola': [
        'coca-cola', 'coca cola', 'coke', 'fanta', 'sprite', 'fuze tea',
        'aquarius', 'minute maid', 'pulco', 'honest', 'innocent', 'powerade',
        'monster energy', 'nestea', 'adez', 'cappy', 'smartwater',
        'the coca cola company'
    ],

    'Kellogg': [
        'kellogg', "kellog's", 'special k', 'all-bran', 'frosties',
        'smacks', 'tresor', 'cheerios', 'cheerios multigrain', 'pringles', 'Pringles', 'PRINGLES'
    ],

    'General Mills': [
        'general mills', 'nature valley', 'old el paso', 'haagen-dazs',
        'häagen-dazs', 'jus rol', 'pillsbury', 'green giant', 'gigante verde'
    ],

    'Kraft Heinz': [
        'heinz', 'kraft', 'orlando', 'hp sauce', 'lea & perrins'
    ],

    'Mars Wrigley': [
        'mars', 'm&m', "m&m's", 'snickers', 'twix', 'orbit', 'skittles',
        'maltesers', 'celebrations', 'reese', 'whiskas', 'pedigree',
        'ebly', "ben's original"
    ],

    'Dr. Oetker': [
        'dr. oetker', 'dr oetker', 'alsa', 'ancel', 'condi', 'muesli crunchy',
        'muesli crunchy'
    ],

    'Deoleo / Aceites': [
        'deoleo', 'carapelli', 'bertolli', 'carbonell', 'koipe', 'hula'
    ],

    'Valeo Foods': [
        'valeo foods', 'balconi', 'kettle', 'kettle chips', 'rowan hill'
    ],

    'Princes Group': [
        'princes', 'napolina', 'juver'
    ],

    'Ecotone / Wessanen': [
        'ecotone', 'wessanen', 'bjorg', 'alter eco',
        'allos', 'clipper', 'tanoshi', 'zonnatura', 'whole earth', 'isitar'
    ],

    'Upfield': [
        'becel', "fruit d'or", 'proactiv', 'pro activ',
        "fruit d'or proactiv,proactiv,proactiv expert,pro activ",
        "fruit d'or,becel", "fruit d'or proactiv, pro activ, proactiv, proactiv expert",
        "fruit d'or proactiv,proactiv,proactiv expert, pro activ"
    ],

    # ======================
    # MARCAS DE DISTRIBUIDOR / RETAILERS
    # ======================

    'Hacendado / Mercadona': [
        'hacendado', 'mercadona', 'deliplus', 'compy', 'bosque verde',
        'entrepinares', 'facendado', 'FACENDADO'
    ],

    'Lidl': [
        'lidl', 'alesto', 'milbona', 'fin carré', 'fin carre', 'sondey',
        'freshona', 'crownfield', 'vemondo', 'vermondo', 'baresa',
        'j.d. gross', 'bellarom', 'combino', 'italiamo', 'snack day',
        'envia', 'nixe', 'maribel', 'saguaro', 'trattoria', 'gelatelli',
        'toque du chef', 'duc de coeur', 'deluxe', "vitad'or",
        "vita d'or", 'belbake', 'primadonna', 'solesita', 'tastino',
        'harvest basket', 'maître jean pierre', 'maitre jean pierre',
        'mister choc', 'favourina', 'favorina'
    ],

    'Carrefour / Système U / Intermarché': [
        'carrefour', 'reflets de france', 'eco +', 'eco+',
        'nos regions ont du talent', 'nos régions ont du talent',
        'nos regions ont du talent', 'nos régions ont du talent',
        'u bio', 'magasins u', 'u ', 'u,', 'u', 'paturages', 'pâturages',
        'páturages', 'paturages, sélection des mousquetaires',
        'sélection des mousquetaires', 'les mousquetaires',
        'nos regions ont du talent', 'nos régions ont du talent',
        'itineraires des saveurs', 'itinéraire des saveurs',
        'itinéraire des saveurs, les mousquetaires',
        'itineraires de nos regions',
        'saveurs de nos regions', 'saveurs de nos régions',
        'saveurs de nos régions', 'saveurs de nos regions'
    ],

    'Leclerc': [
        'e.leclerc', 'bio village', 'marque repère', 'marque repere',
        'saveurs de nos régions', 'saveurs de nos regions'
    ],

    'Retail Internacional (Otros)': [
        'tesco', 'sainsbury', 'waitrose', 'marks & spencer', 'marks and spencer',
        'm&s', 'auchan', 'aldi', 'migros', 'coop', 'picard', 'costco',
        'asda', 'morrisons', 'système u', 'dia', 'el corte inglés',
        'mcennedy', 'stockwell', 'monoprix', 'woolworths food',
        'co op', 'the co-operative loved by us',
        'kirkland', 'kirkland signature', "member's mark"
    ],

    # ======================
    # GRANDES GRUPOS LÁCTEOS / QUESOS
    # ======================

    'Lactalis / Bel': [
        'lactalis', 'président', 'president', 'galbani', 'lactel', 'puleva',
        'lauki', 'société', 'société', 'bridelice', 'bridelight',
        'kiri', 'babybel', 'la vache qui rit', 'boursin', 'leerdammer',
        'candia', 'milsa', 'pilos', 'grand fermage', 'le petit basque',
        'la laitìère', 'la laitière', 'la laitiere', 'nurishh'
    ],

    'Savencia': [
        'savencia', 'st moret', 'saint moret', 'tartare', 'caprice des dieux',
        'saint agur', 'elle & vire', 'elle&vire', 'cœur de lion', 'chavroux',
        'bordeau chesnel', 'le rustique', 'fromarsac', 'rians'
    ],

    'FrieslandCampina': [
        'frieslandcampina', 'milko', 'longley farm'  # milko depende de país, se incluye como aproximación
    ],

    'Arla': [
        'arla', 'castello', 'lurpak'  # si lo tienes, añade otras marcas arla
    ],

    # ======================
    # OTROS GRANDES GRUPOS EUROPEOS
    # ======================

    'Barilla / Pasta': [
        'barilla', 'wasa', 'mulino bianco', 'harry', 'harrys',
        'de cecco', 'rummo', 'panzani', 'giovanni rana', 'rana',
        'misura', 'garofalo', 'jacquet', 'gran cereale', 'gran cereale'
    ],

    'Associated British Foods (ABF)': [
        'jordans', 'dorset cereals', 'twinings', 'ryvita', 'patak',
        'primark', 'allinson', 'kingsmill', 'silver spoon'
    ],

    'Ebro Foods / Mutti': [
        'ebro foods', 'brillante', 'sos', 'lustucru', 'taureau ailé',
        'tilda', 'mutti', 'liebig'
    ],

    'Grandes Grupos Europeos': [
        'fleury michon', 'fleurs michon', 'bonne maman', 'andros',
        'gerblé', 'gerble', 'bonduelle', 'brioche pasquier', 'pasquier',
        'st michel', 'saint michel', 'la boulangère', 'la boulangere',
        'sodebo', 'tipiak', 'gullon', 'gullón', 'cuetara', 'cuétara',
        'mccain', 'mc cain', 'findus', 'bimbo', 'haribo', 'ricola',
        'st hubert', 'paysan breton', 'lotus', 'weetabix', 'pascual',
        'don simón', 'don simon', 'elpozo', 'campofrío', 'campofrio',
        'marie', 'petit navire', 'saupiquet', 'connetable', 'connétable',
        'poulain', 'lucien georgelin', 'pierre martinet', "d'aucy",
        'soignon', 'lesieur', 'quorn', 'entremont', 'william saurin',
        'jaouda', 'chergui', 'aicha', 'cassegrain', 'brets', "bret's",
        'warburtons', 'vico', 'seeberger', 'puget', 'primevère',
        'brossard', 'le gaulois', 'lutti', 'lune de miel', 'francine',
        'le ster', 'coraya', 'storck', 'hovis', 'tramier', 'yeo valley',
        'filippo berio', 'kikkoman', 'saclà', 'lindt', 'sprüngli',
        'bahlsen', 'bahlsen,leibniz', 'leibniz'
    ],

    # ======================
    # BIO / SALUD (GRUPOS GRANDES)
    # ======================

    'Léa Nature / Bio Salud': [
        'jardin bio', 'jardin bio étic', 'lea nature', 'léa nature',
        'karelea', 'bisson', 'evernat', 'ethiquable', 'éthiquable',
        'cereal bio', 'céréal bio', 'sojasun', 'taifun', 'soy',
        'happyvore', 'ecomil', 'eco mil', 'alnatura', 'huel',
        'deliciously ella', 'violife', 'pure via', 'dmbio', 'bio u',
        'isola bio'
    ],

    # ======================
    # BEBIDAS / ALCOHOL
    # ======================

    'Bebidas no alcohólicas (otros grupos)': [
        'red bull', 'redbull', 'schweppes', 'orangina', 'teisseire',
        'cristaline', 'joker', 'pago', 'solan de cabras', 'bezoya',
        'san benedetto', 'oasis', 'robinsons', 'vita coco', 'champomy',
        'mogu mogu', 'mogu mogu (sappé)', 'thanon', 'campo largo',
        'hawai', 'HAWAI', 'champomy'
    ],

    'Cerveceras / Alcohol': [
        'estrella damm', 'estrella galicia', 'heineken', 'peroni',
        'budweiser', 'mahou', '1664', 'leffe', 'grimbergen',
        'tourtel', 'tourtel twist', 'tourtel twist citron',
        'tourtel twist framboise', 'tourtel twist pêche'
    ],

    # ======================
    # EJEMPLOS ADICIONALES DEL ARRAY CON GRUPO CLARO
    # ======================

    'CHO Group (Terra Delyssa)': [
        'terra delyssa', 'terra delyssa estate', 'TERRA DELYSSA'
    ],

    'Ekibio / Le Pain des Fleurs (Léa Compagnie Biodiversité)': [
        'ekibio, le pain des fleurs', 'ekibio,le pain des fleurs',
        'le pain des fleurs', 'le pain des fleurs, ekibio',
        'ekibio', 'le moulin du pivert', 'le moulin du pivert', 'priméal'
    ],

    'La Fournée Dorée (grupo familiar francés)': [
        'la fournée dorée', 'la fournee doree', 'la fournée dorée',
        'la fournée dorée', 'la fournee doree'
    ],

    'Heura Foods': [
        'heura'
    ],

    'Samyang': [
        'samyang', 'buldak,samyang', 'buldak'
    ],

    'Lee Kum Kee': [
        'lee kum kee'
    ],

    'Ayam': [
        'ayam'
    ],

    'Artiach (Grupo Bimbo)': [
        'artiach'
    ],

    'Birds Eye / Nomad Foods': [
        'birds eye'
    ],

    'Mission (Gruma)': [
        'mission', 'gruma,mission wrap'
    ],

    'Mizkan': [
        'mizkan'
    ],

    'Mentos / Perfetti Van Melle': [
        'mentos', 'mentos,perfetti'
    ],

    'Tabasco / McIlhenny': [
        'tabasco', 'mcIlhenny company, tabasco', 'avery island la, mcihenny company, tabasco'
    ],

    'Rügenwalder Mühle': [
        'rügenwalder mühle'
    ],

    'Ritter Sport': [
        'ritter sport', 'RITTER SPORT'
    ],

    'Valor / Nocilla (Idilia Foods)': [
        'valor', 'VALOR', 'nocilla', 'cola cao', 'colacao', 'colacaoidilia foods'
    ]
}


    resultados_dict = {}

    for item in lista_marcas_unicas:
        if pd.isna(item):
            resultados_dict[item] = 'Desconocido'
            continue
        
        item_lower = str(item).lower()
        encontrado = False
        
        for corporacion, marcas in mapping.items():
            # Buscamos si la corporación o alguna de sus marcas está en el texto
            if any(marca in item_lower for marca in marcas) or corporacion.lower() in item_lower:
                resultados_dict[item] = corporacion
                encontrado = True
                break
        
        if not encontrado:
            resultados_dict[item] = 'Otras / Local'

    # 2. Creamos una nueva columna en el DataFrame original usando el mapeo
    # Esto es mucho más eficiente que un bucle for sobre todo el dataset
    return df_columna.map(resultados_dict)

# --- FLUJO DE TRABAJO ---

# Aplicamos la función directamente a la columna de tu dataset original
food['corporacion'] = clasificar_multinacionales(food['brands'])

# Ahora puedes ver el conteo REAL (basado en todas las filas, no solo las únicas)
print("Distribución de marcas en el dataset:")
print(food['corporacion'].value_counts())

# Opcional: Ver qué marcas han quedado como "Otras / Local" para seguir limpiando
print("\nEjemplos de marcas clasificadas como Otras/Local:")
print(food[food['corporacion'] == 'Otras / Local']['brands'].unique()[:10])

Distribución de marcas en el dataset:
corporacion
Carrefour / Système U / Intermarché      1781
Otras / Local                            1697
Grandes Grupos Europeos                   613
Lidl                                      595
Retail Internacional (Otros)              495
Mondelez                                  350
Hacendado / Mercadona                     292
Nestlé                                    259
Danone                                    212
Unilever                                  197
Barilla / Pasta                           142
Ecotone / Wessanen                        136
Coca-Cola                                 120
Kellogg                                   108
Ferrero                                   102
PepsiCo                                    92
Kraft Heinz                                89
Lactalis / Bel                             88
Léa Nature / Bio Salud                     85
Bebidas no alcohólicas (otros grupos)      74
Associated British Foods (ABF)

In [29]:
def obtener_array_no_clasificadas(df, columna_marcas='brands', columna_clasificada='corporacion'):
    """
    Extrae un array de marcas únicas que han quedado como 'Otras / Local',
    ordenadas por su frecuencia de aparición en el dataset.
    """
    # 1. Filtramos el DataFrame para obtener solo las filas no clasificadas
    df_otras = df[df[columna_clasificada] == 'Otras / Local']
    
    # 2. Obtenemos el conteo de apariciones de cada marca
    # Esto nos ayuda a priorizar cuáles añadir al diccionario después
    conteo_marcas = df_otras[columna_marcas].value_counts()
    
    # 3. Convertimos el índice (que son los nombres de las marcas) a una lista/array
    array_no_clasificadas = conteo_marcas.index.tolist()
    
    # Imprimimos información útil para el mentor/alumno
    print(f"Se han encontrado {len(array_no_clasificadas)} marcas únicas sin clasificar.")
    print(f"Las 5 que más aparecen son: {array_no_clasificadas[:5]}")
    
    return array_no_clasificadas

# --- USO ---
# Ejecutamos la función y guardamos el resultado en una variable
marcas_pendientes = obtener_array_no_clasificadas(food)

# Si quieres ver las primeras 50 para copiarlas a tu diccionario:
print(marcas_pendientes)

Se han encontrado 991 marcas únicas sin clasificar.
Las 5 que más aparecen son: ['Yoplait', 'Harvest Morn', "Nairn's", 'Crosta & Mollica', 'Walkers']
['Yoplait', 'Harvest Morn', "Nairn's", 'Crosta & Mollica', 'Walkers', 'Nakd', 'The Foodie Market', 'Specially Selected', 'Kallø', 'Chabrior', 'Graze', 'Schär', 'Rivercote', 'Bio & Me', 'Village Bakery', 'Solevita', 'Proper', 'Bramwells', 'Brooklea', 'Malo', 'Cathedral City', 'Plenish', 'Fage', 'Bisto', 'Loyd Grossman', 'Rowse', 'Grandessa', 'Créaline', 'Worldwide Foods', 'BN', "Green & Black's", 'WW, Weight Watchers', 'Eat Real', 'KP', 'Daddy', 'Star', 'Chabrior, Intermarché', 'Soreen', 'Kara', 'Saladinettes', 'The Spice Tailor', 'Bodymass', 'Branston', 'Strong Roots', 'BOL', 'Organix', 'Linwoods', "Jason's", 'Bridélice', 'Ivoria', 'KIND', 'Parmentier', 'Rigoni di Asiago', 'Valencia', 'Pågen', 'Baxters', 'nākd', 'Rama', 'Terres et Céréales', 'Müller', 'Naked', 'Jif', 'graze', 'Novi', 'Banania', "L'atelier Blini", 'Schar', 'Batts', 'Batche